# AirShift — XGBoost Model Tuning

This notebook focuses on improving the XGBoost model selected during the model comparison stage.

The tuning process will explore different XGBoost hyperparameters and compare the tuned model with the baseline XGBoost model.

The final test period will remain completely unseen during tuning and will only be used for the final evaluation.


## 1. Load the Labeled Dataset

The labeled dataset is loaded from the processed data directory.

This dataset contains the engineered features and the binary deterioration target created in the previous stages of the AirShift pipeline.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_PATH = Path("../data/processed/airshift_labeled.csv")

df = pd.read_csv(DATA_PATH)

df["datetime"] = pd.to_datetime(df["datetime"])

print("Dataset shape:", df.shape)
print("Number of stations:", df["station"].nunique())
print("Date range:", df["datetime"].min(), "to", df["datetime"].max())

Dataset shape: (418381, 100)
Number of stations: 12
Date range: 2013-03-01 00:00:00 to 2017-02-28 17:00:00


In [2]:
print("\nTarget distribution:")
print(df["Deterioration"].value_counts())

print("\nTarget proportions:")
print(df["Deterioration"].value_counts(normalize=True).round(4))


Target distribution:
Deterioration
0.0    219985
1.0    198396
Name: count, dtype: int64

Target proportions:
Deterioration
0.0    0.5258
1.0    0.4742
Name: proportion, dtype: float64


In [3]:
# Quick validation
print("Missing target values:", df["Deterioration"].isna().sum())
print("Duplicate rows:", df.duplicated().sum())

Missing target values: 0
Duplicate rows: 0


## 2. Define Features and Target

The deterioration label is used as the target variable, while identifier and datetime columns are excluded from the model features.

The `datetime` column is retained separately for creating the chronological data splits.


In [4]:
TARGET = "Deterioration"

EXCLUDED_COLUMNS = [
    "Deterioration",
    "No",
    "datetime"
]

X = df.drop(columns=EXCLUDED_COLUMNS)
y = df[TARGET]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Number of features:", X.shape[1])

Feature matrix shape: (418381, 97)
Target shape: (418381,)
Number of features: 97


In [5]:
print("Categorical features:")
print(X.select_dtypes(include=["object"]).columns.tolist())

print("\nNumerical features:")
print(X.select_dtypes(exclude=["object"]).columns.tolist())

Categorical features:
['wd', 'station']

Numerical features:
['year', 'month', 'day', 'hour', 'PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3', 'TEMP', 'PRES', 'DEWP', 'RAIN', 'WSPM', 'day_of_week', 'is_weekend', 'PM2.5_lag_1h', 'PM2.5_lag_3h', 'PM2.5_lag_6h', 'PM10_lag_1h', 'PM10_lag_3h', 'PM10_lag_6h', 'SO2_lag_1h', 'SO2_lag_3h', 'SO2_lag_6h', 'NO2_lag_1h', 'NO2_lag_3h', 'NO2_lag_6h', 'CO_lag_1h', 'CO_lag_3h', 'CO_lag_6h', 'O3_lag_1h', 'O3_lag_3h', 'O3_lag_6h', 'PM2.5_rolling_mean_3h', 'PM2.5_rolling_max_3h', 'PM2.5_rolling_std_3h', 'PM2.5_rolling_mean_6h', 'PM2.5_rolling_max_6h', 'PM2.5_rolling_std_6h', 'PM10_rolling_mean_3h', 'PM10_rolling_max_3h', 'PM10_rolling_std_3h', 'PM10_rolling_mean_6h', 'PM10_rolling_max_6h', 'PM10_rolling_std_6h', 'SO2_rolling_mean_3h', 'SO2_rolling_max_3h', 'SO2_rolling_std_3h', 'SO2_rolling_mean_6h', 'SO2_rolling_max_6h', 'SO2_rolling_std_6h', 'NO2_rolling_mean_3h', 'NO2_rolling_max_3h', 'NO2_rolling_std_3h', 'NO2_rolling_mean_6h', 'NO2_rolling_max_6h', 'NO2_ro

C:\Users\HP\AppData\Local\Temp\ipykernel_6840\2682472949.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print(X.select_dtypes(include=["object"]).columns.tolist())


### Findings

The labeled dataset contains **97 features** after excluding the target, record identifier, and datetime columns. The features include **95 numerical variables** and **2 categorical variables** (`wd` and `station`).
